# Preparing the datasets

Throughout the course, we will use the San Francisco International Airport report on monthly passenger traffic statistics by airline dataset. In this lesson we will:
- Load and review the dataset
- Prepare the dataset to work with AI agents
- Review the DuckDB approach
- Review the PostgreSQL approach 

This dataset is available at [here](https://data.sfgov.org/Transportation/Air-Traffic-Passenger-Statistics/rkru-6vcg/about_data)

## Loading the Air Passenger Traffic Dataset

Let's start by import the required libraries:

In [ ]:
import pandas as pd

In [ ]:
file_path = "../data/Air_Traffic_Passenger_Statistics_20260201.csv"
df = pd.read_csv(file_path)
df.head()

Let's prep the dataset to work wi with AI agents:
- Validate the column names
- Rename the column names
- Remove irrelevant columns
- Reformat the columns


In [ ]:
df["Date"] = pd.to_datetime(df["Activity Period Start Date"], format="%m/%d/%y")

df["Year"] = df["Activity Period"].astype(str).str[:4].astype(int)

columns = [
    "Year",
    "Date",
    "Operating Airline",
    "Operating Airline IATA Code",
    "Published Airline",
    "Published Airline IATA Code",
    "GEO Summary",
    "GEO Region",
    "Activity Type Code",
    "Price Category Code",
    "Terminal",
    "Boarding Area",
    "Passenger Count"
]

air_traffic = df[columns].copy()
air_traffic.dtypes

In [ ]:
air_traffic

In [ ]:
air_traffic.to_csv("../data/air_traffic_gold.csv")

In [ ]:
tbl_name = "air_traffic"

## DuckDB Workflow

In this section, we will review how to set up in-memory DuckDB database using the `ibis` library. Let's start by loading the `ibis` library:

In [ ]:
import ibis

In [ ]:
con_db = ibis.duckdb.connect()
con_db.create_table(tbl_name, air_traffic, overwrite=True)


In [ ]:
con_db.sql("SELECT * FROM air_traffic LIMIT 10").execute()


## Postgres Workflow

In [ ]:
con_postgres = ibis.postgres.connect(
    user="postgres",
    password="password",
    host="postgres",
    port=5432,
    database="my_db",
)

In [ ]:
schema = ibis.memtable(air_traffic).schema()
print(schema)

In [ ]:
con_postgres.create_table("air_traffic", air_traffic, schema=schema, overwrite=True)

In [ ]:
con_postgres.sql("SELECT * FROM air_traffic LIMIT 10").execute()
